# 🔬 Science Paper Analyzer — полное руководство по кодуДобро пожаловать! Этот ноутбук — подробное, построчное объяснение каждого файла нашего проекта.> **Как пользоваться:** просто читайте. Код не нужно запускать (хотя можете — он работает).> > Ноутбук открывается по ссылке:> https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/tutorial.ipynb## Структура проекта```science-paper-analyzer/├── science_paper_analyzer.ipynb  ← Colab-лаунчер (три шага)├── tutorial.ipynb                ← этот файл├── analyzer.py                   ← центральный координатор├── parsers/│   ├── utils.py                  ← extract_year() — общая функция│   ├── arxiv_parser.py           ← парсер arXiv│   ├── semantic_scholar.py       ← парсер Semantic Scholar│   ├── openreview_parser.py      ← парсер OpenReview│   ├── aclanthology.py           ← парсер ACL Anthology│   ├── jmlr_parser.py            ← парсер JMLR│   ├── crossref_fallback.py      ← универсальный парсер (CrossRef)│   └── cyberleninka.py           ← парсер CyberLeninka├── analyzers/│   ├── citation_analyzer.py      ← анализ цитируемости│   ├── text_analyzer.py          ← AI-детектор текста│   └── journal_analyzer.py       ← проверка журнала (Beall's List)├── exporters/│   ├── csv_exporter.py           ← экспорт в CSV│   └── excel_exporter.py         ← экспорт в Excel (с цветами)├── app.py                        ← веб-интерфейс (Streamlit)├── requirements.txt              ← зависимости├── ROAD.md                       ← документация разработки└── README.md                     ← инструкция по запуску```Поехали!

---# 1️⃣ Главный файл: science_paper_analyzer.ipynb---Это **точка входа** для пользователя. Она открывается в Google Colab и состоит из 3 шагов.## Шаг 1: Клонирование + установка```pythonimport subprocess, sys, os, importlibREPO_URL = "https://github.com/DmitPerson42/science-paper-analyzer.git"REPO_DIR = "science-paper-analyzer"# Удаляем старую копию (чтобы всегда была свежая версия)if os.path.exists(REPO_DIR):    import shutil    shutil.rmtree(REPO_DIR, ignore_errors=True)# Клонируем репозиторийsubprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True, ...)os.chdir(REPO_DIR)# Устанавливаем только недостающие пакеты (чтобы не переустанавливать всё)required = ["requests", "pandas", "openpyxl", "lxml", "beautifulsoup4", "numpy", "scikit-learn"]missing = [p for p in required if importlib.util.find_spec(p) is None]if missing:    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, ...)print("✅ Шаг 1 выполнен!")```### Построчный разбор:Строка за строкой:1. `import subprocess, sys, os, importlib` — загружаем системные модули. `subprocess` для запуска команд (`git clone`, `pip install`). `importlib.util.find_spec` чтобы проверить, установлен ли пакет.2. `REPO_URL = "https://github.com/..."` — откуда качать код.3. `REPO_DIR = "science-paper-analyzer"` — имя папки после клонирования.4. `if os.path.exists(REPO_DIR):` — если папка уже есть (от предыдущего запуска).5. `import shutil / shutil.rmtree(...)` — удаляем её со всем содержимым. Это гарантирует, что вы всегда работаете с последней версией кода.6. `subprocess.run(["git", "clone", ...])` — клонируем репозиторий. `--depth 1` значит без истории коммитов (быстрее).7. `os.chdir(REPO_DIR)` — переходим внутрь папки с проектом.8. `required = [...]` — список нужных библиотек.9. `importlib.util.find_spec(p)` — проверяет, импортируется ли пакет. Если нет — он в списке `missing`.10. `subprocess.run([... "pip", "install", "-q"] + missing, ...)` — устанавливаем то, чего не хватает. `-q` для тихого режима.11. `print("✅ Шаг 1 выполнен!")` — сигнал пользователю, что можно переходить к шагу 2.

## Шаг 2: Запуск анализа```pythonimport sys, pandas as pdsys.path.insert(0, ".")from analyzer import PaperAnalyzerQUERY = "filtering generative text data language model collapse"MAX_PER_SOURCE = 5analyzer = PaperAnalyzer(verbose=True)papers = analyzer.collect_papers(QUERY, max_per_source=MAX_PER_SOURCE)if not papers:    print("Статей не найдено")else:    df = analyzer.analyze_papers(papers)    display(df)  # показываем таблицу    # + статистика```### Построчный разбор:1. `import sys, pandas as pd` — стандартные библиотеки.2. `sys.path.insert(0, ".")` — добавляем текущую папку в пути поиска модулей. Без этого Python не найдёт файл `analyzer.py`.3. `from analyzer import PaperAnalyzer` — импортируем наш главный класс.4. `QUERY = "..."` — тема поиска (можете изменить на свою).5. `MAX_PER_SOURCE = 5` — сколько статей брать с каждого источника.6. `analyzer = PaperAnalyzer(verbose=True)` — создаём экземпляр. `verbose=True` включает подробный вывод в консоль Colab.7. `papers = analyzer.collect_papers(...)` — запускаем сбор статей со всех 10 источников.8. `if not papers:` — если ничего не нашли — сообщаем.9. `df = analyzer.analyze_papers(papers)` — анализируем все статьи по трём критериям. Возвращает pandas DataFrame.10. `display(df_view)` — красивая таблица с результатами.

## Шаг 3: Экспорт```pythonif "df" in dir() and len(df) > 0:    csv_path = "analysis_results.csv"    xlsx_path = "analysis_results.xlsx"    analyzer.export_csv(df, csv_path)    analyzer.export_excel(df, xlsx_path)    from google.colab import files    files.download(csv_path)  # браузер скачает файл```1. `if "df" in dir() and len(df) > 0:` — проверяем, что переменная `df` существует и не пустая.2. `analyzer.export_csv(df, csv_path)` — сохраняем CSV на диск Colab.3. `analyzer.export_excel(df, xlsx_path)` — сохраняем Excel с цветами.4. `from google.colab import files / files.download(...)` — скачиваем файл на компьютер пользователя через браузер.

---# 2️⃣ parsers/utils.py — extract_year()---```pythonimport refrom datetime import datetime, timezonedef extract_year(value) -> str:    if value is None or value == "":        return ""    # Если число — проверяем границы    if isinstance(value, (int, float)):        if 1900 <= value <= 2050:            return str(int(value))        # Если большое число — это timestamp (мс или сек)        if value > 10**9:            if value > 10**12:                value = value / 1000  # конвертируем мс → сек            try:                return str(datetime.fromtimestamp(value, tz=timezone.utc).year)            except:                return ""        return ""    # Строка    s = str(value).strip()    match = re.search(r'(19[0-9][0-9]|20[0-9][0-9])', s)    if match:        return match.group(1)    return ""```### Зачем это нужно?Разные источники возвращают год в разном формате:- **arXiv**: строку вида `"2023-01-15"` — надо извлечь `"2023"`- **OpenReview**: число `1652000000000` (миллисекунды timestamp) — надо конвертировать- **Semantic Scholar**: число `2023` — просто в строку- **CyberLeninka**: строку вида `"2023 год"` — надо извлечь `"2023"`Функция `extract_year()` решает всё это сразу.### Построчный разбор:1. `import re` — регулярные выражения для поиска года в строке.2. `from datetime import datetime, timezone` — для конвертации timestamp.3. `if value is None or value == "":` — пустые значения сразу возвращают пустую строку.4. `isinstance(value, (int, float)):` — проверка, пришло ли число.5. `if 1900 <= value <= 2050:` — если число похоже на год (например, `2023`), возвращаем как строку.6. `if value > 10**9:` — если число больше миллиарда, это скорее всего timestamp.7. `if value > 10**12:` — если 13+ цифр, это миллисекунды. Делим на 1000.8. `datetime.fromtimestamp(value, tz=timezone.utc).year` — конвертируем timestamp в год (например, `1652000000000 → 2022`).9. Для строки: `re.search(r'\b(19\d\d|20\d\d)\b', s)` — ищем 4 цифры от 1900 до 2099, окружённые границами слова.

---# 3️⃣ parsers/arxiv_parser.py — парсер arXiv---```pythonimport requests, time, xml.etree.ElementTree as ETfrom typing import List, Dictfrom parsers.utils import extract_yearARXIV_API = "https://export.arxiv.org/api/query"class ArxivParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {"search_query": f"all:{query}", "start": 0, "max_results": max_results}        headers = {"User-Agent": "SciencePaperAnalyzer/1.0"}        # Повторные попытки при ошибках        max_retries = 3        for attempt in range(max_retries):            try:                resp = requests.get(ARXIV_API, params=params, headers=headers, timeout=30)                if resp.status_code == 429:  # Too Many Requests                    time.sleep((attempt + 1) * 3)                    continue                resp.raise_for_status()                break            except requests.exceptions.HTTPError:                if attempt == max_retries - 1:                    return []                time.sleep(2)        time.sleep(1)  # Вежливая пауза        # Парсим XML-ответ        root = ET.fromstring(resp.text)        ns = {"a": "http://www.w3.org/2005/Atom"}        papers = []        for entry in root.findall("a:entry", ns):            title = entry.findtext("a:title", "").strip().replace("\n", " ")            summary = entry.findtext("a:summary", "").strip().replace("\n", " ")            published = entry.findtext("a:published", "")            link_el = entry.find("a:id", ns)            link = link_el.text if link_el is not None else ""            authors = [a.findtext("a:name", "") for a in entry.findall("a:author", ns) if a.findtext("a:name", "")]            year = extract_year(published)            papers.append({                "id": link.split("/")[-1],       # arXiv ID                "title": title,                "authors": ", ".join(authors[:5]),  # первые 5 авторов                "year": year,                "abstract": summary,                "url": link,                "venue": "arXiv",            })        return papers```### Построчный разбор:1. `import xml.etree.ElementTree as ET` — стандартный модуль для парсинга XML. arXiv возвращает XML (не JSON).2. `ARXIV_API = "https://export.arxiv.org/api/query"` — эндпоинт API arXiv.3. `params = {"search_query": f"all:{query}", "start": 0, "max_results": max_results}` — `all:` значит искать во всех полях (title, abstract, authors).4. **Retry-логика**: если сервер отвечает `429` (слишком много запросов), ждём 3-6-9 секунд и пробуем снова.5. `resp.raise_for_status()` — если статус не 200, вызовет исключение.6. `time.sleep(1)` — вежливая пауза перед следующим запросом.7. `ET.fromstring(resp.text)` — парсим XML.8. `ns = {"a": "http://www.w3.org/2005/Atom"}` — пространство имён Atom, в котором arXiv возвращает данные.9. `entry.findtext("a:title", "")` — находим текст внутри тега `<a:title>`. Если тега нет — возвращаем `""`.10. `.replace("\n", " ")` — удаляем переносы строк внутри заголовка/аннотации.11. `entry.findall("a:author")` — все теги `<a:author>`. Из каждого извлекаем `<a:name>`.12. `link.split("/")[-1]` — из URL вида `http://arxiv.org/abs/2301.12345` берём `2301.12345`.13. `authors[:5]` — ограничиваем 5 авторами, чтобы не раздувать таблицу.

---# 4️⃣ parsers/semantic_scholar.py — парсер Semantic Scholar---```pythonimport requestsfrom typing import List, DictSEMANTIC_API = "https://api.semanticscholar.org/graph/v1/paper/search"FIELDS = "title,authors,year,citationCount,abstract,venue,externalIds,url"class SemanticScholarParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {"query": query, "limit": max_results, "fields": FIELDS}        headers = {"User-Agent": "SciencePaperAnalyzer/1.0"}        resp = requests.get(SEMANTIC_API, params=params, headers=headers, timeout=30)        if resp.status_code == 429:            raise Exception("Rate limited. Please wait and try again.")        resp.raise_for_status()        data = resp.json()        papers = []        for paper in data.get("data", []):            authors = [a.get("name", "") for a in paper.get("authors", [])]            ext_ids = paper.get("externalIds", {}) or {}            papers.append({                "id": paper.get("paperId", ""),                "title": paper.get("title", "N/A"),                "authors": ", ".join(authors[:5]),                "year": str(paper.get("year", "")) if paper.get("year") else "",                "abstract": paper.get("abstract", "") or "",                "url": f"https://www.semanticscholar.org/paper/{paper.get('paperId', '')}",                "citation_count": paper.get("citationCount", 0),                "venue": paper.get("venue", ""),                "doi": ext_ids.get("DOI", ""),            })        return papers```### Построчный разбор:1. `FIELDS = "title,authors,year,citationCount,..."` — говорим API, какие поля хотим получить. Если не указать, вернётся только ID.2. `params = {"query": query, "limit": max_results, "fields": FIELDS}` — стандартный GET-запрос к REST API.3. `data = resp.json()` — Semantic Scholar возвращает JSON.4. `data.get("data", [])` — список статей лежит в ключе `"data"`.5. `paper.get("authors", [])` — авторы приходят как список словарей: `[{"name": "Иван Петров"}, ...]`.6. `a.get("name", "") for a in ...` — из каждого словаря берём `"name"`.7. `ext_ids = paper.get("externalIds", {}) or {}` — внешние ID (DOI, ArXiv ID). Если API вернул `null` — превращаем в пустой словарь.8. `citation_count` — ключевое поле! Semantic Scholar — единственный источник, дающий количество цитирований бесплатно.9. Проверка `if paper.get("year")` нужна, потому что у старых статей может не быть года.

---# 5️⃣ parsers/openreview_parser.py — парсер OpenReview---```pythonimport requestsfrom typing import List, Dictfrom parsers.utils import extract_yearOPENREVIEW_API = "https://api.openreview.net/notes/search"class OpenReviewParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {"term": query, "limit": max_results, "source": "forum"}        # Попытка поиска        try:            resp = requests.get(OPENREVIEW_API, params=params, ..., timeout=30)            resp.raise_for_status()        except Exception:            # Fallback: другой эндпоинт            alt_url = f"https://api.openreview.net/notes?term={query}&limit={max_results}"            resp = requests.get(alt_url, ..., timeout=30)            resp.raise_for_status()        data = resp.json()        notes = data.get("notes", data.get("rows", []))        papers = []        for note in notes[:max_results]:            content = note.get("content", {})            title = content.get("title", note.get("title", "N/A"))            if isinstance(title, dict):                title = title.get("value", "N/A")  # OpenReview wraps in {"value": "..."}            authors = content.get("authors", note.get("authors", []))            if isinstance(authors, dict):                authors = authors.get("value", [])            abstract = content.get("abstract", "")            if isinstance(abstract, dict):                abstract = abstract.get("value", "")            forum = note.get("forum", "")            cdate = note.get("cdate", note.get("tcdate", 0))            year = extract_year(cdate)  # <-- исправление бага с 1652 годом            papers.append({"title": title, "year": year, ...})        return papers```### Построчный разбор:1. **Проблема с годом**: OpenReview возвращает `cdate` как миллисекунды: `1652000000000`.    - Старый код: `str(cdate)[:4]` → `"1652"` ❌   - Новый код: `extract_year(cdate)` → `"2022"` ✅2. **Два эндпоинта**: основной `/notes/search` иногда падает, поэтому есть fallback `/notes`.3. **Вложенная структура**: OpenReview хранит метаданные в `content` как словарь с ключом `"value"`. Проверка `isinstance(title, dict)` нужна, чтобы безопасно извлечь `title["value"]`.4. `notes.get("cdate", notes.get("tcdate", 0))` — `cdate` это время создания, `tcdate` это время создания темы. Современные OpenReview notes имеют оба поля.

---# 6️⃣ parsers/aclanthology.py — парсер ACL Anthology---```pythonimport requestsfrom typing import List, DictACL_API = "https://api.aclanthology.org/v1/search"class ACLAnthologyParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {"q": query, "limit": max_results}        try:            resp = requests.get(ACL_API, params=params, ..., timeout=30)            resp.raise_for_status()        except Exception:            return self._scrape_search(query, max_results)  # fallback        data = resp.json()        results = data.get("results", data.get("data", []))        papers = []        for paper in results[:max_results]:            anth_id = paper.get("paper_id", paper.get("id", ""))            title = paper.get("title", "N/A")            authors_list = paper.get("authors", [])            year = str(paper.get("year", ""))            abstract = paper.get("abstract", "")            papers.append({                "id": anth_id,                "title": title,                "authors": ", ".join(authors_list[:5]),                "year": year,                "abstract": abstract,                "url": f"https://aclanthology.org/{anth_id}/",                "venue": "ACL Anthology",            })        return papers```### Построчный разбор:1. API ACL Anthology принимает `?q=query&limit=N`.2. `data.get("results", data.get("data", []))` — разные версии API возвращают по-разному.3. `paper.get("paper_id", paper.get("id", ""))` — в ответе может быть `paper_id` или просто `id`.4. `_scrape_search()` — fallback на HTML scraping, если API недоступен.5. `anth_id` — это идентификатор вида `2023.acl-long.1`, по которому можно собрать URL.

---# 7️⃣ parsers/jmlr_parser.py — парсер JMLR---```pythonimport requests, refrom bs4 import BeautifulSoupJMLR_URL = "https://jmlr.org/papers/"def _jmlr_volume_to_year(href: str) -> str:    # JMLR URL: https://jmlr.org/papers/v22/i1.html    match = re.search(r'/v(\d+)', href)    if not match:        return ""    vol = int(match.group(1))    # v1 = 2000, v2 = 2001, ..., v22 = 2021    year = 2000 + vol - 1    return str(year) if 1999 <= year <= 2030 else ""class JMLRParser:    def search(self, query, max_results=5):        resp = requests.get(JMLR_URL, ...)        soup = BeautifulSoup(resp.text, "lxml")        links = soup.select("a[href*='papers/v']")        for link in links:            href = link.get("href", "")            title = link.get_text(strip=True)            # фильтр по ключевым словам            if query.lower() not in title.lower():                continue            papers.append({                "year": _jmlr_volume_to_year(href),  # v22 → 2021                ...            })        return papers```### Построчный разбор:1. У JMLR **нет API**, только HTML-страница. Используем `BeautifulSoup` для парсинга.2. `soup.select("a[href*='papers/v']")` — ищем все ссылки, содержащие `papers/v` (ссылки на статьи).3. `_jmlr_volume_to_year(href)` — JMLR нумерует тома с v1=2000. URL вида `/papers/v22/i1.html` → том 22 → год 2021.4. `if query.lower() not in title.lower(): continue` — фильтруем только статьи, содержащие ключевые слова.5. `link.get_text(strip=True)` — извлекаем текст ссылки (название статьи).

---# 8️⃣ parsers/crossref_fallback.py — универсальный парсер (CrossRef)---```pythonimport requestsfrom typing import List, DictCROSSREF_API = "https://api.crossref.org/works"class CrossRefFallback:    def __init__(self, source_name: str = "CrossRef"):        self.source_name = source_name  # например "ResearchGate", "eLibrary"    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {            "query": query,            "rows": max_results,            "select": "title,author,DOI,URL,abstract,publication-date,publisher,container-title"        }        headers = {"User-Agent": "SciencePaperAnalyzer/1.0 (mailto:example@example.com)"}        try:            resp = requests.get(CROSSREF_API, params=params, ..., timeout=30)            resp.raise_for_status()        except Exception as e:            return self._empty_result(query, str(e))        data = resp.json()        items = data.get("message", {}).get("items", [])        papers = []        for item in items[:max_results]:            title = item.get("title", ["N/A"])[0]            authors_list = item.get("author", [])            authors = ", ".join([                f"{a.get('given', '')} {a.get('family', '')}".strip()                for a in authors_list[:5]            ])            doi = item.get("DOI", "")            # Извлекаем год из дат публикации            date_parts = item.get("published-online", {}).get("date-parts") or                         item.get("published-print", {}).get("date-parts") or                         item.get("issued", {}).get("date-parts") or []            year = str(date_parts[0][0]) if date_parts and date_parts[0] else ""            papers.append({                "id": doi, "title": title, "authors": authors,                "year": year, "doi": doi,                "venue": item.get("container-title", [""])[0],                "source_context": self.source_name,  # <-- важно!            })        if not papers:            return self._empty_result(query, "No results from CrossRef")        return papers```### Построчный разбор:1. **Зачем этот парсер?** ResearchGate, eLibrary, Dissercat, FIPS не имеют открытого API. CrossRef — центральный реестр научных публикаций с бесплатным API.2. `__init__(self, source_name)` — мы создаём **4 экземпляра** этого класса с разными `source_name`, чтобы пользователь видел, для какого источника искали.3. `"select": "title,author,DOI,..."` — просим только нужные поля (экономит трафик).4. `data.get("message", {}).get("items", [])` — структура ответа CrossRef: `{"message": {"items": [...]}}`.5. `item.get("title", ["N/A"])[0]` — заголовок приходит как список (массив) из одного элемента.6. `f"{a.get('given', '')} {a.get('family', '')}".strip()` — сборка ФИО из given name + family name.7. **Извлечение года**: CrossRef возвращает дату в виде `[[2023, 1, 15]]`. Берём первый элемент первого массива.8. `_empty_result()` — если ничего не нашли, возвращаем 1 статью-заглушку с пояснением.

---# 9️⃣ parsers/cyberleninka.py — парсер CyberLeninka---```pythonimport requestsfrom bs4 import BeautifulSoupfrom typing import List, Dictfrom parsers.utils import extract_yearCYBERLENINKA_BASE = "https://cyberleninka.ru"class CyberLeninkaParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        headers = {"User-Agent": "Mozilla/5.0 ... AppleWebKit/537.36"}        try:            url = f"{CYBERLENINKA_BASE}/search?q={query}"            resp = requests.get(url, headers=headers, timeout=30)            resp.raise_for_status()        except Exception as e:            return self._fallback_empty(query, str(e))        soup = BeautifulSoup(resp.text, "lxml")        articles = soup.select("article, .search-item, .b-serp-item")        if not articles:            articles = soup.select("li.result-item, div.result, .b-article")        for article in articles[:max_results]:            title_el = article.select_one("a[href*='/article/'], a[href*='article']")            if not title_el:                title_el = article.select_one("h2 a, h3 a, .title a")            href = title_el.get("href", "")            if href and not href.startswith("http"):                href = CYBERLENINKA_BASE + href            title = title_el.get_text(strip=True) if title_el else "N/A"            year_el = article.select_one(".year, .date, .b-serp-item__date")            year = year_el.get_text(strip=True) if year_el else ""            papers.append({                "title": title,                "year": extract_year(year),  # "2023 год" → "2023"                ...            })        return papers```### Построчный разбор:1. User-Agent содержит `Mozilla/5.0 ...`, чтобы CyberLeninka не блокировала как бота.2. `soup.select(...)` — CSS-селекторы для поиска карточек статей. CyberLeninka использует несколько шаблонов страниц, поэтому перечисляем несколько вариантов.3. `article.select_one("a[href*='/article/']")` — ищем ссылку, в URL которой есть `/article/`.4. `if href and not href.startswith("http"): href = CYBERLENINKA_BASE + href` — относительные ссылки превращаем в абсолютные.5. `extract_year(year)` — CyberLeninka может вернуть `"2023"` или `"2023 год"` — функция корректно обработает оба варианта.

---# 🔟 analyzers/citation_analyzer.py — анализ цитируемости---```pythonclass CitationAnalyzer:    def analyze(self, paper: dict) -> Tuple[float, str]:        # Пытаемся найти citation_count из любого поля        citation_count = paper.get("citation_count", None)        if citation_count is None:            citation_count = paper.get("citations", None)        if citation_count is None:            citation_count = paper.get("cited_by_count", None)        year_str = paper.get("year", "")        try:            year = int(year_str) if year_str and year_str.isdigit() else None        except:            year = None        # Если нет данных о цитированиях        if citation_count is None:            return 0.5, "Нет данных — оценка нейтральная"        citation_count = int(citation_count)        current_year = datetime.now().year        # Логика оценки        if year and (current_year - year) >= 3 and citation_count == 0:            return 0.2, "Подозрительно: статья старше 3 лет без цитирований"        elif citation_count == 0:            return 0.5, "Нейтрально: молодая статья без цитирований"        elif citation_count <= 5:            return 0.7, f"{citation_count} цит. — приемлемо"        elif citation_count <= 15:            return 0.85, f"{citation_count} цит. — хорошо"        else:            return 0.95, f"{citation_count} цит. — высокий уровень"```### Построчный разбор:1. `paper.get("citation_count", None)` — Semantic Scholar возвращает это поле. Другие парсеры могут использовать `"citations"` или `"cited_by_count"`.2. `int(year_str) if year_str and year_str.isdigit()` — год должен состоять только из цифр. Если это строка "N/A" — не парсим.3. **Логика оценки:**   - Статья **>=3 года** с **0 цитирований** → score 0.2 (подозрительно). Хорошая статья за 3 года наберёт хотя бы пару цитирований.   - **0 цитирований** у новой статьи → score 0.5 (нейтрально). Может, просто ещё не успели процитировать.   - **1-5** → score 0.7. Нормальный уровень.   - **5-15** → score 0.85. Хорошо цитируемая работа.   - **>15** → score 0.95. Высокий уровень доверия.

---# 1️⃣1️⃣ analyzers/text_analyzer.py — AI-детектор текста---```pythonclass TextAnalyzer:    def __init__(self):        self.ai_markers = [            "as an ai", "as a language model", "i cannot", "i'm sorry",            "it is important to note", "in conclusion", "additionally",            "furthermore", "in recent years", "state-of-the-art",            "cutting-edge", "groundbreaking", "leveraging",        ]    def analyze(self, paper: dict) -> Tuple[float, str]:        text = " ".join(str(paper.get(f, "")) for f in ["title", "abstract", "summary"])        text = text.strip()        if len(text) < 50:            return 0.5, "Недостаточно текста для анализа"        words = text.split()        text_lower = text.lower()        # 1. AI-маркеры: считаем, сколько фраз-маркеров встречается        marker_hits = sum(1 for m in self.ai_markers if m in text_lower)        marker_score = 1.0 - min(marker_hits / len(self.ai_markers) * 3, 0.8)        # 2. Лексическое разнообразие (TTR)        unique_words = set(w.lower() for w in words)        ttr = len(unique_words) / len(words)        # TTR = 0.65 идеален для научного текста        ttr_score = 1.0 - abs(ttr - 0.65) / 0.65        # 3. Повторяющиеся 3-граммы        repeat_penalty = 0.0        if len(words) > 10:            seen_ngrams = set()            for i in range(len(words) - 2):                ngram = " ".join(words[i:i+3]).lower()                if ngram in seen_ngrams:                    repeat_penalty += 0.02                seen_ngrams.add(ngram)        repeat_penalty = min(repeat_penalty, 0.5)        # 4. Равномерность длины предложений        sent_lengths = [len(s.split()) for s in re.split(r'[.!?]+', text) if len(s.split()) > 2]        if sent_lengths:            avg = sum(sent_lengths) / len(sent_lengths)            std = (sum((l - avg)**2 for l in sent_lengths) / len(sent_lengths)) ** 0.5            rel_std = std / max(avg, 1)            uniformity_penalty = 0.2 if rel_std < 0.3 else (0.1 if rel_std > 1.2 else 0.0)        else:            uniformity_penalty = 0.0        # Итог: взвешенная сумма        score = (marker_score * 0.3 + ttr_score * 0.2 +                 (1.0 - repeat_penalty) * 0.25 +                 (1.0 - uniformity_penalty) * 0.1 +                 (1.0 - stop_penalty) * 0.15)        return max(0.0, min(1.0, score)), detail_message```### Построчный разбор:Этот модуль — **эвристический детектор AI-сгенерированного текста**. Он не использует нейросети (они недоступны офлайн), а применяет 5 лингвистических метрик:1. **AI-маркеры**: фразы вроде `"as an AI language model"`, `"I cannot"`, `"in recent years"` — AI-модели часто их используют. Чем больше таких фраз, тем ниже score.2. **Type-Token Ratio (TTR)**: отношение уникальных слов к общему количеству слов. У людей TTR выше (более разнообразный словарь), у AI — ниже (больше повторов). Идеал для научного текста — 0.65.3. **Повторяющиеся 3-граммы**: скользящее окно из 3 слов. Если одни и те же тройки слов встречаются часто — это признак шаблонного (AI) текста.4. **Равномерность длины предложений**: AI пишет предложения примерно одинаковой длины. Люди — то коротко, то длинно.5. **Стоп-слова**: AI использует больше стоп-слов (`the`, `a`, `an`, `and`, `of`, `to`). У людей соотношение более вариативно.Каждая метрика даёт свой score от 0 до 1. Финальный score — взвешенная сумма (веса подобраны эмпирически).

---# 1️⃣2️⃣ analyzers/journal_analyzer.py — проверка журнала (Beall's List)---```python# Список хищнических журналов (~180 наименований)PREDATORY_JOURNALS = [    "international journal of engineering research and technology",    "journal of emerging technologies and innovative research",    "science publishing group",    "waset (world academy of science, engineering and technology)",    ...  # всего ~180]# Список доверенных площадокTRUSTED_VENUES = [    "nature", "science", "ieee", "acl", "neurips", "icml",    "arxiv", "jmlr", "springer", "elsevier", "acm",    "openreview", "semantic scholar",    ...]class JournalAnalyzer:    def __init__(self):        self.predatory_set = set(PREDATORY_JOURNALS)  # set для быстрого поиска        self.trusted_set = set(v.lower() for v in TRUSTED_VENUES)    def analyze(self, paper: dict) -> Tuple[float, str]:        # Собираем все поля, которые могут содержать название площадки        venue_text = " ".join([            str(paper.get("venue", "")),            str(paper.get("source", "")),            str(paper.get("publisher", "")),        ]).lower().strip()        # Проверка доверенных        for trusted in self.trusted_set:            if trusted in venue_text:                return 0.95, f"Площадка '{venue}' доверенная"        # Проверка хищнических        for predatory in self.predatory_set:            if predatory in venue_text:                return 0.15, f"Журнал найден в Beall's List (хищнический)"        return 0.6, "Журнал не найден в Beall's List — нейтрально"```### Построчный разбор:1. `PREDATORY_JOURNALS` — **Beall's List** (сокращённая версия ~180 журналов). Полный список содержит более 1200 наименований. Это журналы, которые публикуют статьи за деньги без рецензирования.2. `TRUSTED_VENUES` — проверенные площадки. Если статья в Nature или на arXiv — доверие максимальное.3. `self.predatory_set = set(...)` — конвертируем список в **множество (set)** для O(1) поиска.4. **venue_text**: объединяем все поля, которые могут содержать название площадки. Один парсер может вернуть venue как `"OpenReview"`, другой — как `"ACL Anthology"`.5. `if trusted in venue_text` — **частичное совпадение**. Если в тексте есть "nature communications", а в списке есть "nature" — найдёт.6. `return 0.95, "..."` — если площадка доверенная, сразу возвращаем высокий score.7. `return 0.15, "..."` — если журнал хищнический, возвращаем низкий score.8. `return 0.6, "..."` — если не нашли ни там, ни там — нейтральная оценка.

---# 1️⃣3️⃣ exporters/csv_exporter.py — экспорт в CSV---```pythonimport pandas as pdclass CSVExporter:    def export(self, df: pd.DataFrame, filepath: str):        # Отбираем только основные колонки для CSV        export_cols = [            "title", "authors", "year", "source", "url",            "citation_score", "text_score", "journal_score",            "overall_score", "verdict"        ]        export_cols = [c for c in export_cols if c in df.columns]  # защита от отсутствия колонки        df_export = df[export_cols].copy()        df_export.to_csv(filepath, index=False, encoding="utf-8-sig")        return filepath```### Построчный разбор:1. `export_cols = [c for c in export_cols if c in df.columns]` — оставляем только те колонки, которые реально есть в таблице. Если какой-то анализатор не сработал — не будет ошибки.2. `df[export_cols].copy()` — берём только нужные колонки (без деталей-пояснений, они есть только в Excel).3. `to_csv(..., encoding="utf-8-sig")` — `utf-8-sig` добавляет BOM (Byte Order Mark) в начале файла. Это нужно, чтобы **Excel в Windows** правильно открыл русские буквы.4. `index=False` — не сохраняем индекс (номера строк).

---# 1️⃣4️⃣ exporters/excel_exporter.py — экспорт в Excel (с цветами)---```pythonfrom openpyxl import Workbookfrom openpyxl.styles import PatternFill, Font, Borderfrom openpyxl.utils.dataframe import dataframe_to_rowsclass ExcelExporter:    def export(self, df: pd.DataFrame, filepath: str):        wb = Workbook()        ws = wb.active        ws.title = "Results"        # Те же колонки, но + детали        export_cols = ["title", "authors", "year", "source", "url",                       "citation_score", "text_score", "journal_score",                       "overall_score", "verdict",                       "citation_detail", "text_detail", "journal_detail"]        df_export = df[export_cols].copy()        for row in dataframe_to_rows(df_export, index=False, header=True):            ws.append(list(row))        # Стили для заголовка        header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")        header_font = Font(color="FFFFFF", bold=True, size=11)        for cell in ws[1]:            cell.fill = header_fill            cell.font = header_font        # Цвета для вердикта        green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  # Real        yellow_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")  # Suspicious        red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")     # Fake        # Ищем колонку "verdict" и красим        for row in ws.iter_rows(min_row=2, max_col=ws.max_column, max_row=ws.max_row):            for cell in row:                if "Real" in str(cell.value):                    cell.fill = green_fill                elif "Suspicious" in str(cell.value):                    cell.fill = yellow_fill                elif "Fake" in str(cell.value):                    cell.fill = red_fill        # Ширина колонок        ws.column_dimensions["A"].width = 60   # title        ws.column_dimensions["B"].width = 40   # authors        wb.save(filepath)```### Построчный разбор:1. `openpyxl` — библиотека для создания Excel файлов с форматированием.2. `dataframe_to_rows(df_export, ...)` — конвертирует pandas DataFrame в строки для openpyxl.3. **Цветовое кодирование**: зелёный = Real, жёлтый = Suspicious, красный = Fake.4. `PatternFill(start_color="C6EFCE", ... fill_type="solid")` — определяет цвет заливки ячейки.5. `ws.iter_rows(min_row=2, ...)` — проходим по всем строкам, начиная со второй (первая — заголовок).6. `column_dimensions["A"].width = 60` — задаём ширину колонок, чтобы названия статей не обрезались.

---# 1️⃣5️⃣ analyzer.py — центральный координатор---Это **самый важный файл**. Он собирает всё вместе.## Импорты и Streamlit-заглушка```pythonimport pandas as pd, timefrom datetime import datetime# Streamlit нужен только для веб-версии.# Для Colab/консоли создаём заглушку.try:    import streamlit as st    _has_streamlit = Trueexcept ImportError:    class _Stub:        def progress(self, *a, **kw): return self        def warning(self, msg): print(f"[WARNING] {msg}")        def __getattr__(self, name): return lambda *a, **kw: None    st = _Stub()    _has_streamlit = False```### Построчный разбор этой части:1. `try: import streamlit as st` — Streamlit — библиотека для веб-интерфейса. В Colab её нет.2. `except ImportError:` — если не установлен, создаём **класс-заглушку**.3. `class _Stub:` — все методы (`progress()`, `empty()`, `warning()`) просто ничего не делают или печатают в консоль.4. `st = _Stub()` — теперь код может вызывать `st.progress(0)` в любом режиме без ошибок.## PaperAnalyzer.collect_papers()```pythondef collect_papers(self, query, max_per_source=5):    all_papers = []    for source_key, parser in self.parsers.items():        try:            papers = parser.search(query, max_results=max_per_source)            for p in papers:                p["source"] = source_key                # генерация ID, если его нет                if "id" not in p or not p["id"]:                    p["id"] = f"{source_key}_{hash(p.get('title', '')) % 100000}"            all_papers.extend(papers)            time.sleep(0.3)  # вежливая задержка между запросами        except Exception as e:            self._log(f"[WARNING] {source_key}: {e}")    # Дедупликация    seen = set()    unique_papers = []    for p in all_papers:        key = p.get("title", "").strip().lower()[:100]        if key and key not in seen:            seen.add(key)            unique_papers.append(p)    return unique_papers```### Построчный разбор:1. `self.parsers.items()` — это 10 парсеров (arxiv, semantic_scholar, ..., fips).2. `parser.search(...)` — каждый парсер возвращает список статей.3. `p["source"] = source_key` — добавляем метку, откуда пришла статья (например, `"arxiv"`).4. `if "id" not in p or not p["id"]:` — если парсер не вернул ID, генерируем свой.5. `time.sleep(0.3)` — 300ms пауза между запросами к разным источникам, чтобы не заблокировали.6. **Дедупликация**: `seen = set()` — запоминаем уникальные названия. Если статья с таким же названием уже есть — пропускаем.7. `key = title[:100].lower()` — берём первые 100 символов названия в нижнем регистре.

## PaperAnalyzer.analyze_papers()```pythondef analyze_papers(self, papers: list) -> pd.DataFrame:    results = []    for idx, paper in enumerate(papers):        # Запускаем 3 анализатора        citation_score, citation_detail = self.analyzers["citation"].analyze(paper)        text_score, text_detail = self.analyzers["text"].analyze(paper)        journal_score, journal_detail = self.analyzers["journal"].analyze(paper)        scores = [citation_score, text_score, journal_score]        avg_score = sum(scores) / len(scores)  # среднее арифметическое        # Итоговый вердикт        if avg_score >= 0.7:            verdict = "Real"        elif avg_score >= 0.4:            verdict = "Suspicious"        else:            verdict = "Fake"        results.append({            "title": paper.get("title", "N/A"),            "year": paper.get("year", "N/A"),            "citation_score": round(citation_score, 2),            "verdict": verdict,            # ... ещё 15 полей        })    return pd.DataFrame(results)```### Построчный разбор:1. `self.analyzers["citation"].analyze(paper)` — передаём всю информацию о статье в анализатор. Каждый анализатор возвращает (score, detail). 2. `avg_score = sum(scores) / 3` — среднее арифметическое трёх scores.3. **Вердикт:**   - ≥ 0.7 → Real (достоверная)   - 0.4–0.7 → Suspicious (подозрительная)   - < 0.4 → Fake (фейковая)4. `round(citation_score, 2)` — округляем до 2 знаков после запятой.5. `pd.DataFrame(results)` — возвращаем таблицу, с которой удобно работать в pandas.

---# 1️⃣6️⃣ app.py — веб-интерфейс (Streamlit)---`app.py` — это **Streamlit-приложение**. Оно запускается командой `streamlit run app.py` и открывается в браузере.### Ключевые части:```pythonimport streamlit as stfrom analyzer import PaperAnalyzer, SOURCE_DISPLAY# Боковая панельwith st.sidebar:    query = st.text_input("Тема поиска:")    max_sources = st.slider("Источники", 1, 10, 10)    max_per_source = st.slider("Статей/источник", 1, 10, 5)    # Чекбоксы для каждого источника    for key in all_sources[:max_sources]:        st.checkbox(SOURCE_DISPLAY[key], value=True)    if st.button("🚀 Запустить анализ"):        analyzer = PaperAnalyzer()        papers = analyzer.collect_papers(query, max_per_source)        df = analyzer.analyze_papers(papers)        st.session_state.results_df = df# Основная областьif st.session_state.results_df is not None:    df = st.session_state.results_df    # Статистика    st.metric("✅ Достоверные", real_count)    st.metric("⚠️ Подозрительные", sus_count)    st.metric("❌ Фейковые", fake_count)    # Таблица с подсветкой    styled = df.style.applymap(color_verdict, subset=["verdict"])    st.dataframe(styled)    # Детальный просмотр    selected = st.selectbox("Выберите статью:", df["title"])    # 3 вкладки: Обзор, Анализ, Текст```### Построчный разбор:1. `st.set_page_config(layout="wide")` — растягиваем страницу на всю ширину.2. `st.session_state` — Streamlit-переменная, которая хранится между перезапусками скрипта. Позволяет сохранить результаты после нажатия кнопки.3. `st.sidebar` — боковая панель с настройками.4. `st.checkbox(SOURCE_DISPLAY[key], value=True)` — чекбоксы для выбора источников (все включены по умолчанию).5. `st.progress()` — прогресс-бар при сборе статей.6. `st.spinner("🔍 Сбор...")` — анимация загрузки.7. `styled = df.style.applymap(color_verdict, subset=["Вердикт"])` — применяем функцию, которая возвращает CSS-стиль в зависимости от значения.8. `st.tabs(["📄 Обзор", "📊 Анализ", "📝 Текст"])` — три вкладки для детального просмотра.9. `st.success()` / `st.warning()` / `st.error()` — цветные блоки для вердикта (зелёный/жёлтый/красный).

---# 1️⃣7️⃣ requirements.txt — зависимости---```streamlit>=1.28.0    # веб-интерфейсrequests>=2.28.0     # HTTP-запросы к APIpandas>=1.5.0        # таблицы и DataFrameopenpyxl>=3.0.0      # Excel файлыlxml>=4.9.0          # быстрый XML/HTML парсерbeautifulsoup4>=4.11.0  # HTML scrapingnumpy>=1.23.0        # численные расчётыscikit-learn>=1.2.0  # (запас) ML-моделиplotly>=5.14.0       # (запас) графики```### Для чего каждая?- `streamlit` — чтобы запустить `app.py` (веб-версия). Для Colab не нужен.- `requests` — сердце парсеров. Все 10 источников опрашиваются через `requests.get()`.- `pandas` — результаты анализа возвращаются как DataFrame. Удобно фильтровать, сортировать, экспортировать.- `openpyxl` — для создания .xlsx с цветной разметкой.- `lxml` + `beautifulsoup4` — парят HTML CyberLeninka и JMLR, а также XML от arXiv.- `numpy` + `scikit-learn` + `plotly` — запасные библиотеки для возможного расширения (ML-модели, графики).> **В Colab** эти библиотеки уже предустановлены. Ноутбук проверяет через `importlib.util.find_spec()` и устанавливает только то, чего нет.

---# 🎓 Итог: как всё работает вместе## Диаграмма потока данных```Вы вводите тему (например, "model collapse")         │         ▼┌──────────────────────┐│  PaperAnalyzer        ││  .collect_papers()    │└──────┬──────┬──────┬──┘       │      │      │       ▼      ▼      ▼┌──────┐┌──────┐┌──────────┐│arXiv ││OpenRev││CrossRef  │  ← 10 парсеров│  XML ││  JSON││  JSON    │└──┬───┘└──┬───┘└────┬─────┘   │      │         │   ▼      ▼         ▼┌──────────────────────────┐│  Список словарей (статьи) ││  [{"title": "...",        ││    "authors": "...",      ││    "citation_count": 5},  ││   {...}, ...]             │└──────────┬───────────────┘           │           ▼┌──────────────────────────┐│  PaperAnalyzer            ││  .analyze_papers()        │└──────┬──────┬──────┬──────┘       │      │      │       ▼      ▼      ▼┌────────┐┌──────┐┌──────┐│Citation││ Text ││Journl│  ← 3 анализатора│Analyzer││Analyz││Analyz│└───┬────┘└──┬───┘└──┬───┘    │       │       │    ▼       ▼       ▼┌──────────────────────────┐│  pandas DataFrame         ││  с scores и вердиктом     │└──────────┬───────────────┘           │           ▼┌──────────┴──────────┐│  Вывод/Экспорт       ││                      ││  ┌──────┐ ┌────────┐ ││  │CSV   │ │ Excel  │ ││  │.csv  │ │.xlsx   │ ││  │      │ │(цвета) │ ││  └──────┘ └────────┘ │└─────────────────────┘```## Как интерпретировать результаты| Score | Вердикт | Что значит ||-------|---------|------------|| ≥ 0.70 | **Real** ✅ | Статья цитируется, текст естественный, площадка доверенная. Всё хорошо. || 0.40–0.70 | **Suspicious** ⚠️ | Что-то не так: мало цитирований, или текст похож на AI, или журнал неизвестный. Стоит проверить вручную. || < 0.40 | **Fake** ❌ | Серьёзные проблемы: нет цитирований при возрасте >3 лет, журнал в Beall's List, текст явно сгенерирован. |## Что делать, если всё показывается "Fake"?1. **Убедитесь, что тема на английском** — русскоязычные статьи хуже индексируются.2. **Проверьте соединение с интернетом** — без него парсеры не работают.3. **Посмотрите детали** в колонках `citation_detail`, `text_detail`, `journal_detail` — там написано, почему именно снизился score.4. **Увеличьте MAX_PER_SOURCE** — может, просто мало данных.

---# 📚 Полезные ссылки- **Репозиторий GitHub:** https://github.com/DmitPerson42/science-paper-analyzer- **Запустить анализ (Colab):** https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/science_paper_analyzer.ipynb- **Этот туториал:** https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/tutorial.ipynb- **Тема диссертации:** Разработка методов фильтрации генеративных текстовых данных для предотвращения коллапса языковых моделей---*Создано в рамках диссертационной работы. 2025.*